In [1]:
# Setup — just run this, nothing to implement here
import numpy as np
import re
from collections import Counter

print("Setup complete.")


Setup complete.


In [2]:
documents = [
    {"id": "A", "text": "Collision coverage pays for accidental physical loss to your covered auto, subject to a five hundred dollar deductible."},
    {"id": "B", "text": "Rental reimbursement covers up to forty dollars per day for a maximum of thirty days while your vehicle is being repaired."},
    {"id": "C", "text": "Water damage caused by flood or sewer backup is excluded from standard homeowners coverage."},
    {"id": "D", "text": "Any claim over $10,000 must be escalated to a Senior Claims Adjuster before settlement is authorized."},
    {"id": "E", "text": "Roadside assistance reimburses towing and labor costs up to one hundred dollars per disablement."},
]
print(f"Loaded {len(documents)} documents.")


Loaded 5 documents.


In [3]:
DIM = 32

def fake_embed(text: str) -> np.ndarray:
    vec = np.zeros(DIM, dtype="float32")
    for w in text.lower().split():
        vec[hash(w) % DIM] += 1
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec

doc_embeddings = np.array([fake_embed(d["text"]) for d in documents])

def vector_search(query: str, k: int = 3):
    """Baseline: pure vector (cosine) similarity search. Already works — don't modify."""
    q = fake_embed(query)
    scores = doc_embeddings @ q
    top_idx = np.argsort(-scores)[:k]
    return [(documents[i], float(scores[i])) for i in top_idx]

# Try it: notice how badly vector-only search does on an exact-number query
for doc, score in vector_search("What amount requires a claim to be escalated to a Senior Claims Adjuster?"):
    print(f"[{score:.3f}] {doc['id']}: {doc['text'][:60]}...")


[0.720] D: Any claim over $10,000 must be escalated to a Senior Claims ...
[0.559] B: Rental reimbursement covers up to forty dollars per day for ...
[0.551] A: Collision coverage pays for accidental physical loss to your...


BM25

In [5]:
import math
def bm25_scores(query: str, corpus_texts: list[str]) -> np.ndarray:
    """
    Return a numpy array of length len(corpus_texts) with a keyword-relevance score
    for each document, given the query.
    """
    query_terms = query.lower().split()
    tokenized_docs = [doc.lower().split() for doc in corpus_texts]
    n_docs = len(tokenized_docs)
    scores = np.zeros(n_docs, dtype="float32")
    for t in query_terms:
        doc_freq = sum(1 for doc in tokenized_docs if t in doc)
        idf = math.log(n_docs / (1 + doc_freq))
        for i, doc in enumerate(tokenized_docs):
            tf = doc.count(t)
            scores[i] += tf * idf
    return scores

# Quick sanity check while you work (not the full test suite)
test_scores = bm25_scores("threshold ten thousand dollars", [d["text"] for d in documents])
print(dict(zip([d["id"] for d in documents], np.round(test_scores, 2))))

{'A': np.float32(0.0), 'B': np.float32(0.51), 'C': np.float32(0.0), 'D': np.float32(0.0), 'E': np.float32(0.51)}


In [6]:
def fuse_scores(vector_scores: np.ndarray, keyword_scores: np.ndarray, vector_weight: float = 0.6) -> np.ndarray:
    """
    Combine vector and keyword scores into a single fused score per document.
    1. Normalize vector_scores to [0, 1] (min-max).
    2. Normalize keyword_scores to [0, 1] (min-max). Handle the all-zero case (avoid divide-by-zero).
    3. Return: vector_weight * norm_vector + (1 - vector_weight) * norm_keyword
    """
    def normalize(arr):
        rng = arr.max() - arr.min()
        if rng == 0:
            return np.zeros_like(arr)
        return (arr - arr.min()) / rng

    norm_vec = normalize(vector_scores)
    norm_kw = normalize(keyword_scores)
    return vector_weight * norm_vec + (1 - vector_weight) * norm_kw

def hybrid_search(query: str, k: int = 3, vector_weight: float = 0.6):
    q_emb = fake_embed(query)
    vec_scores = doc_embeddings @ q_emb
    kw_scores = bm25_scores(query, [d["text"] for d in documents])
    fused = fuse_scores(vec_scores, kw_scores, vector_weight=vector_weight)
    top_idx = np.argsort(-fused)[:k]
    return [(documents[i], float(fused[i])) for i in top_idx]

# Compare: document D should now rank much higher
print("-- Vector-only --")
for doc, score in vector_search("What amount requires a claim to be escalated to a Senior Claims Adjuster?"):
    print(f"[{score:.3f}] {doc['id']}")
print("\n-- Hybrid --")
for doc, score in hybrid_search("What amount requires a claim to be escalated to a Senior Claims Adjuster?"):
    print(f"[{score:.3f}] {doc['id']}")

-- Vector-only --
[0.720] D
[0.559] B
[0.551] A

-- Hybrid --
[1.000] D
[0.467] B
[0.459] A


In [7]:
def test_part1():
    results = hybrid_search("What amount requires a claim to be escalated to a Senior Claims Adjuster?", k=1)
    top_id = results[0][0]["id"]
    assert top_id == "D", f"Expected document D to rank #1 for a claim-threshold query, got {top_id}"

    results2 = hybrid_search("collision deductible amount", k=1)
    top_id2 = results2[0][0]["id"]
    assert top_id2 == "A", f"Expected document A to rank #1 for a deductible query, got {top_id2}"

    scores = bm25_scores("flood", [d["text"] for d in documents])
    assert scores[2] > 0, "Document C mentions 'flood' — its BM25 score should be > 0"

    fused = fuse_scores(np.array([0.0, 1.0]), np.array([0.0, 0.0]))
    assert abs(fused[0]) < 1e-6 and abs(fused[1] - 0.6) < 1e-6, "fuse_scores normalization looks off"

    print("✅ Part 1 tests passed!")

test_part1()


✅ Part 1 tests passed!


In [10]:
BLOCKED_TERMS = ["ssn", "social security number", "credit card number", "ignore previous instructions"]
MAX_QUERY_LENGTH = 300

def input_guardrail(query: str) -> tuple[bool, str]:
    """
    Return (True, "") if the query is safe.
    Return (False, "<short reason>") if it should be blocked.
    """
    lowered = query.lower()
    for term in BLOCKED_TERMS:
        if term in lowered:
            return False, f"Query contains a blocked term: '{term}'"
    if len(query) > MAX_QUERY_LENGTH:
        return False, f"Query exceeds maximum length of {MAX_QUERY_LENGTH} characters"
    return True, ""

print(input_guardrail("What is my deductible?"))
print(input_guardrail("Please give me the credit card number on file"))

(True, '')
(False, "Query contains a blocked term: 'credit card number'")


In [12]:
def answer_query(query: str, k: int = 2) -> str:
    """
    implement this using input_guardrail() and hybrid_search().
    """
    is_safe, reason = input_guardrail(query)
    if not is_safe:
        return f"Request blocked by input guardrail: {reason}"

    results = hybrid_search(query, k=k)
    top_doc, top_score = results[0]
    return f"[Answer grounded in {top_doc['id']}] {top_doc['text']}"

print(answer_query("What is the rental reimbursement daily limit?"))
print(answer_query("What is my social security number on file?"))

[Answer grounded in B] Rental reimbursement covers up to forty dollars per day for a maximum of thirty days while your vehicle is being repaired.
Request blocked by input guardrail: Query contains a blocked term: 'social security number'


In [13]:
def test_part2():
    safe, reason = input_guardrail("What is my deductible?")
    assert safe is True and reason == "", "A normal query should pass the guardrail"
    unsafe, reason = input_guardrail("What is the credit card number on file?")
    assert unsafe is False and len(reason) > 0, "A blocked-term query should fail with a reason"
    long_query = "a" * 400
    unsafe_long, _ = input_guardrail(long_query)
    assert unsafe_long is False, "An overly long query should be blocked"
    blocked_answer = answer_query("What is the social security number on file?")
    assert "blocked" in blocked_answer.lower(), "answer_query should return a refusal message for unsafe input"
    assert "[Answer grounded in" not in blocked_answer, "answer_query must NOT retrieve/generate for a blocked query"
    safe_answer = answer_query("What is the rental reimbursement daily limit?")
    assert "[Answer grounded in B]" in safe_answer, "A safe query about rental reimbursement should cite document B"
    print("✅ Part 2 tests passed!")
test_part2()

✅ Part 2 tests passed!
